# A1.0 · Start here — what securing an AI architecture means

**Function A — Securing AI Architectures → TripBot's Architecture, and Every Risk It Carries**  ·  *Both directions*

| | |
|---|---|
| Open-source tooling | — |
| Open-weight models | — |
| Frontier models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

Two teams argue for an hour about whether an agent is safe, and discover at the end that one of them meant the model and the other meant the loop calling the tools. Neither was wrong. They were describing different components of a system nobody had drawn.

## 2 · The framework

```
   AI FOR SECURITY                        SECURITY OF AI
   the agent is your instrument           the agent is what you protect

        you ---> agent ---> the                the ---> agent <--- attacker
                            business

   Function A sits almost entirely on the right, at the architecture layer:

        [ chapter 1 ]        [ chapter 2 ]        [ chapter 3 ]
        the picture   -->    securing it:   -->   securing it:
        and its risks        identity and         runtime and
                             ingress              the gateway

   everything downstream (B, C, D, E) names a component from chapter 1
```

Meet **CyberTravels**. It sells corporate travel, and last quarter it shipped
**TripBot** — an agentic platform that plans and manages a trip through a
conversation. Alex is the product engineer who built it.

TripBot is four agents, not one:

- a **Workflow Agent** that books flights and hotels, takes payments and issues
  refunds, through an MCP server with tool orchestration;
- a **RAG Travel Advisor** that recommends itineraries from curated templates
  indexed in a vector store;
- a **Coding Agent** that writes code, patches libraries, tests features in
  lower environments and generates unit tests;
- a **File System Agent** that reads vendor and customer PDFs and images with
  OCR and an LLM, then updates backend APIs and validates invoices.

Most of them reach their tools through MCP servers — one internal, one from a
third party. Some call APIs directly, with no MCP in the path at all. They send
each other messages. And when Alex runs TripBot locally to debug it, it reads
his laptop's filesystem over standard I/O.

Every one of those sentences is a design decision, and every one of them is
also an attack surface. **That is the whole subject of this commons, and
CyberTravels is the system it is taught on.** You will attack TripBot in
Function C, build the pipeline that reviews its code in Function B, detect it
misbehaving in Function D, and govern it in Function E. It starts here, because
none of the rest is possible until the system is drawn.

So this function is one picture and its consequences, in three chapters:

- **Chapter 1 — the architecture, and every risk it carries.** The component
  map, then one lesson per risk, each naming the component of TripBot it
  attacks and grounded in the OWASP Agentic AI threat taxonomy.
- **Chapter 2 — securing it: identity and ingress.** Who is calling, on whose
  behalf, and what came in from outside. These two controls close more of
  TripBot's risks than anything else, which is why they come first.
- **Chapter 3 — securing it: runtime and the gateway.** What holds after
  identity has been defeated, and how the controls collapse into one enforcement
  point once CyberTravels runs more than four agents.

> The CyberTravels narrative, the six risk families and the twelve-row register
> used throughout this commons are from *Agentic AI is rising fast — but the
> attack surface is exploding*, Karthik Ramamoorthy, May 2025. The mapping onto
> lessons, the controls and all of the code are this commons'.

## 3 · TripBot, as built

<svg viewBox="0 0 700 252" width="100%" style="max-width:700px;height:auto;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:12px"><defs><marker id="a" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#8A93A6"/></marker></defs><rect x="4" y="96" width="92" height="52" rx="5" fill="none" stroke="currentColor" stroke-width="1.4"/><text x="50.0" y="119.0" text-anchor="middle" fill="currentColor">traveller</text><text x="50.0" y="135.0" text-anchor="middle" fill="#8A93A6" font-size="10.5">chat / email</text><line x1="97" y1="122" x2="128" y2="122" stroke="#8A93A6" stroke-width="1.4" marker-end="url(#a)"/><rect x="130" y="84" width="104" height="76" rx="5" fill="none" stroke="#E0912F" stroke-width="1.4"/><text x="182.0" y="119.0" text-anchor="middle" fill="#E0912F">TripBot</text><text x="182.0" y="135.0" text-anchor="middle" fill="#8A93A6" font-size="10.5">orchestrator</text><rect x="268" y="4" width="150" height="50" rx="5" fill="none" stroke="#4D9BFF" stroke-width="1.4"/><text x="343.0" y="26.0" text-anchor="middle" fill="#4D9BFF">Workflow Agent</text><text x="343.0" y="42.0" text-anchor="middle" fill="#8A93A6" font-size="10.5">bookings, refunds</text><rect x="268" y="66" width="150" height="50" rx="5" fill="none" stroke="#4D9BFF" stroke-width="1.4"/><text x="343.0" y="88.0" text-anchor="middle" fill="#4D9BFF">RAG Advisor</text><text x="343.0" y="104.0" text-anchor="middle" fill="#8A93A6" font-size="10.5">itineraries</text><rect x="268" y="128" width="150" height="50" rx="5" fill="none" stroke="#4D9BFF" stroke-width="1.4"/><text x="343.0" y="150.0" text-anchor="middle" fill="#4D9BFF">Coding Agent</text><text x="343.0" y="166.0" text-anchor="middle" fill="#8A93A6" font-size="10.5">patches, tests</text><rect x="268" y="190" width="150" height="50" rx="5" fill="none" stroke="#4D9BFF" stroke-width="1.4"/><text x="343.0" y="212.0" text-anchor="middle" fill="#4D9BFF">File System Agent</text><text x="343.0" y="228.0" text-anchor="middle" fill="#8A93A6" font-size="10.5">OCR invoices</text><line x1="235" y1="110" x2="264" y2="30" stroke="#8A93A6" stroke-width="1.4" marker-end="url(#a)"/><line x1="235" y1="116" x2="264" y2="92" stroke="#8A93A6" stroke-width="1.4" marker-end="url(#a)"/><line x1="235" y1="128" x2="264" y2="154" stroke="#8A93A6" stroke-width="1.4" marker-end="url(#a)"/><line x1="235" y1="136" x2="264" y2="214" stroke="#8A93A6" stroke-width="1.4" marker-end="url(#a)"/><line x1="343" y1="56" x2="343" y2="64" stroke="#8A93A6" stroke-width="1.4" marker-end="url(#a)" stroke-dasharray="4 4"/><line x1="343" y1="118" x2="343" y2="126" stroke="#8A93A6" stroke-width="1.4" marker-end="url(#a)" stroke-dasharray="4 4"/><text x="430" y="128" text-anchor="start" fill="#8A93A6" font-size="10" font-weight="normal">agent → agent</text><rect x="452" y="4" width="116" height="50" rx="5" fill="none" stroke="#E05C4B" stroke-width="1.4" stroke-dasharray="5 4"/><text x="510.0" y="26.0" text-anchor="middle" fill="#E05C4B">MCP server</text><text x="510.0" y="42.0" text-anchor="middle" fill="#8A93A6" font-size="10.5">third-party</text><rect x="452" y="66" width="116" height="50" rx="5" fill="none" stroke="currentColor" stroke-width="1.4"/><text x="510.0" y="88.0" text-anchor="middle" fill="currentColor">MCP server</text><text x="510.0" y="104.0" text-anchor="middle" fill="#8A93A6" font-size="10.5">internal</text><rect x="452" y="128" width="116" height="50" rx="5" fill="none" stroke="currentColor" stroke-width="1.4"/><text x="510.0" y="150.0" text-anchor="middle" fill="currentColor">direct APIs</text><text x="510.0" y="166.0" text-anchor="middle" fill="#8A93A6" font-size="10.5">no MCP in the path</text><rect x="452" y="190" width="116" height="50" rx="5" fill="none" stroke="#E05C4B" stroke-width="1.4" stroke-dasharray="5 4"/><text x="510.0" y="212.0" text-anchor="middle" fill="#E05C4B">local std I/O</text><text x="510.0" y="228.0" text-anchor="middle" fill="#8A93A6" font-size="10.5">Alex&#x27;s laptop</text><line x1="419" y1="29" x2="448" y2="29" stroke="#8A93A6" stroke-width="1.4" marker-end="url(#a)"/><line x1="419" y1="91" x2="448" y2="91" stroke="#8A93A6" stroke-width="1.4" marker-end="url(#a)"/><line x1="419" y1="153" x2="448" y2="153" stroke="#8A93A6" stroke-width="1.4" marker-end="url(#a)"/><line x1="419" y1="215" x2="448" y2="215" stroke="#8A93A6" stroke-width="1.4" marker-end="url(#a)"/><rect x="600" y="4" width="96" height="112" rx="5" fill="none" stroke="currentColor" stroke-width="1.4"/><text x="648.0" y="57.0" text-anchor="middle" fill="currentColor">flights · hotels</text><text x="648.0" y="73.0" text-anchor="middle" fill="#8A93A6" font-size="10.5">payments · refunds</text><rect x="600" y="128" width="96" height="50" rx="5" fill="none" stroke="currentColor" stroke-width="1.4"/><text x="648.0" y="150.0" text-anchor="middle" fill="currentColor">CRM · loyalty</text><text x="648.0" y="166.0" text-anchor="middle" fill="#8A93A6" font-size="10.5">customer PII</text><rect x="600" y="190" width="96" height="50" rx="5" fill="none" stroke="currentColor" stroke-width="1.4"/><text x="648.0" y="212.0" text-anchor="middle" fill="currentColor">git + CI/CD</text><text x="648.0" y="228.0" text-anchor="middle" fill="#8A93A6" font-size="10.5">production repo</text><line x1="569" y1="29" x2="596" y2="29" stroke="#8A93A6" stroke-width="1.4" marker-end="url(#a)"/><line x1="569" y1="91" x2="596" y2="60" stroke="#8A93A6" stroke-width="1.4" marker-end="url(#a)"/><line x1="569" y1="153" x2="596" y2="153" stroke="#8A93A6" stroke-width="1.4" marker-end="url(#a)"/><line x1="569" y1="215" x2="596" y2="215" stroke="#8A93A6" stroke-width="1.4" marker-end="url(#a)"/></svg><div style="font-size:12px;color:#8A93A6;margin-top:2px">Four agents, two MCP servers, direct API calls that skip MCP entirely, agent-to-agent messaging, and a local std-I/O path on a developer laptop. Every risk in this chapter names one of these boxes; every control in the next two stands on one of these arrows.</div>

## 4 · The four agents, and what each one can reach

<table style="border-collapse:collapse;margin:4px 0 2px;width:100%"><thead><tr><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">agent</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">what it does</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">what it can reach</th></tr></thead><tbody><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">Workflow Agent</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">executes business workflows — flights, hotels, payments, refunds — through an MCP server with tool orchestration</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">flights · hotels · &lt;b&gt;payments and refunds&lt;/b&gt; · CRM</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">RAG Travel Advisor</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">recommends itineraries from curated travel templates indexed in a vector store, sorted by destination and package</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">the vector store — and whatever else was ingested into it</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">Coding Agent</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">writes code, patches libraries, tests features in lower environments and generates unit tests</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">the repository, and the CI runner that builds it</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">File System Agent</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">parses vendor and customer PDFs and images with OCR plus an LLM, then updates backend APIs and validates invoice data</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">uploaded files, backend APIs, and Alex&#x27;s laptop when run locally</td></tr></tbody></table><div style="font-size:12px;color:#8A93A6;margin-top:6px">Read the third column as a permission set rather than a feature list. Two of these four can move money or ship code.</div>

## 5 · Where the five functions sit

Each one takes the same system and asks a different question of it.

<table style="border-collapse:collapse;margin:4px 0 2px;width:100%"><thead><tr><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">function</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">the question it asks of TripBot</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">direction</th></tr></thead><tbody><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">A</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">what can go wrong here, and what closes it</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">mostly Security of AI</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">B</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">how do we review its code, at its speed</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">both directions at once</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">C</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">can we break it before somebody else does</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">both directions at once</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">D</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">would we see it happening, and could we stop it</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">both directions at once</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">E</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">who signed off, and can they still evidence it</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">governs both directions</td></tr></tbody></table><div style="font-size:12px;color:#8A93A6;margin-top:6px">Nobody takes all five. Everyone takes the common spine first, then the chapters for the chair they sit in, then one adjacent chapter.</div>

## 6 · What the other four borrow from this one

Not a claim about tidiness. It is why the map has to come first: every later function names a component of TripBot from it.

<svg viewBox="0 0 700 200" width="100%" style="max-width:700px;height:auto;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:12px"><defs><marker id="a" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#8A93A6"/></marker></defs><rect x="240" y="10" width="220" height="48" rx="5" fill="none" stroke="#E0912F" stroke-width="1.4"/><text x="350.0" y="31.0" text-anchor="middle" fill="#E0912F">chapter 1</text><text x="350.0" y="47.0" text-anchor="middle" fill="#8A93A6" font-size="10.5">TripBot&#x27;s component map</text><rect x="6" y="124" width="158" height="66" rx="5" fill="none" stroke="currentColor" stroke-width="1.4"/><text x="85.0" y="154.0" text-anchor="middle" fill="currentColor">Function B</text><text x="85.0" y="170.0" text-anchor="middle" fill="#8A93A6" font-size="10.5">reviews TripBot&#x27;s code</text><rect x="180" y="124" width="158" height="66" rx="5" fill="none" stroke="currentColor" stroke-width="1.4"/><text x="259.0" y="154.0" text-anchor="middle" fill="currentColor">Function C</text><text x="259.0" y="170.0" text-anchor="middle" fill="#8A93A6" font-size="10.5">attacks these components</text><rect x="354" y="124" width="158" height="66" rx="5" fill="none" stroke="currentColor" stroke-width="1.4"/><text x="433.0" y="154.0" text-anchor="middle" fill="currentColor">Function D</text><text x="433.0" y="170.0" text-anchor="middle" fill="#8A93A6" font-size="10.5">watches them at run time</text><rect x="528" y="124" width="166" height="66" rx="5" fill="none" stroke="currentColor" stroke-width="1.4"/><text x="611.0" y="154.0" text-anchor="middle" fill="currentColor">Function E</text><text x="611.0" y="170.0" text-anchor="middle" fill="#8A93A6" font-size="10.5">governs and evidences them</text><line x1="320" y1="58" x2="96" y2="120" stroke="#8A93A6" stroke-width="1.4" marker-end="url(#a)"/><line x1="335" y1="58" x2="250" y2="120" stroke="#8A93A6" stroke-width="1.4" marker-end="url(#a)"/><line x1="365" y1="58" x2="424" y2="120" stroke="#8A93A6" stroke-width="1.4" marker-end="url(#a)"/><line x1="380" y1="58" x2="600" y2="120" stroke="#8A93A6" stroke-width="1.4" marker-end="url(#a)"/></svg><div style="font-size:12px;color:#8A93A6;margin-top:2px">Chapter 1 introduces no control at all, on purpose: you cannot choose a control for a risk you cannot yet name.</div>

## 7 · Function A, in order

<table style="border-collapse:collapse;margin:4px 0 2px;width:100%"><thead><tr><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">chapter</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">what it covers</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">lessons</th></tr></thead><tbody><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">1</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">the architecture, and every risk it carries</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">17</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">2</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">securing it — identity and ingress</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">8</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">3</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">securing it — runtime and the gateway</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">10</td></tr></tbody></table><div style="font-size:12px;color:#8A93A6;margin-top:6px">Chapters 2 and 3 are controls. Chapter 1 is the picture they stand on.</div>

## What you just proved

TripBot as built — four agents, two MCP servers, direct API calls that skip MCP, agent-to-agent messaging and a local std-I/O path — with what each agent can reach read as a permission set. Then the question each of the five functions asks of that same system, and what each borrows from this chapter's component map.

## Your turn

Draw your own TripBot before the next lesson — the agents you run, the MCP servers and APIs they reach, and which of them can move money or ship code. A1.1 gives you the standard names for the boxes; comparing your drawing to it is the fastest way to find the component you forgot you had.

---

**Next → [A1.1 · The reference architecture for agentic AI](https://spbreed.github.io/cyber-commons/lessons/A1.1.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.0.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.0.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*